* refactor, extract code for training a DL - retriever model
* Run after 

In [ ]:
import pandas as pd
import re
import os
from tensorflow.keras.layers import Dot, Activation
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers import Adam
import tensorflow as tf
# import tensorflow_recommenders as tfrs

# 👉 everything below comes from *tf.keras*
from tensorflow import keras
from tensorflow.keras.utils   import FeatureSpace
from tensorflow.keras.layers  import TextVectorization

from keras_rs.layers import BruteForceRetrieval
from keras_rs.metrics import PrecisionAtK, RecallAtK
from keras.losses import BinaryFocalCrossentropy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model            import LogisticRegression
from sklearn.metrics                 import roc_auc_score

import numpy as np, pandas as pd, tensorflow as tf, keras
from keras.layers import TextVectorization, Embedding, Concatenate

# import tensorflow_recommenders as tfrs          # 0.7.3+
import tensorflow as tf#, keras
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

from sklearn.model_selection import StratifiedGroupKFold,RepeatedStratifiedKFold, StratifiedGroupKFold
from dl_model_def import make_fs, TwoTowerDual, build_two_tower_model#, build_tower
from dl_model_def import build_tower
%load_ext autoreload
%autoreload 2

In [ ]:
TRAIN_DL = True
SAVE_MODEL =  False 
# GET_ALL_PREDS = True#False
# SAVE_PREDS = True
GET_ALL_PREDS = True
SAVE_PREDS = False

RUN_CV = True
# RUN_CV = False

EMB_ID= 64

# functions

In [ ]:
from sklearn.metrics import classification_report
import numpy as np
import tensorflow as tf

def evaluate_model(
        model,
        test_df        : pd.DataFrame,
        disease_df     : pd.DataFrame,
        recommend_fn = None,
        k              : int   = 10,
        batch_size     : int   = 4096,
        disease_id_col : str   = "diseaseId",
        disease_name_col: str  = "name",
        drop_known     : bool  = False,          # ← keep / drop known positives
        AltCutoff      : float = 0.6,
        print_reports  : bool  = True,           # NEW: control printing
        compute_topk   : bool  = False,           # NEW: control top-k eval
):
    """
    Evaluate the model on test_df.

    Returns a dict with:
      - 'auc', 'pr_auc', 'bce'
      - 'accuracy', 'precision', 'recall', 'f1'
      - optionally precision@k / recall@k if compute_topk=True
    """

    # ---------- 1. classification head: build dataset & get AUC/PR-AUC/BCE ---------- #
    def _make_ds(df):
        feats = {
            "query": {
                "disease_text": df["disease_text"],
                "diseaseId":    df["diseaseId"],
            },
            "candidate": {
                "target_text":  df["target_text"],
                "targetId":     df["targetId"],
            },
        }
        y = {
            "cls":   df["label"].astype("float32"),
            "score": df["score"].astype("float32"),
        }
        return tf.data.Dataset.from_tensor_slices((feats, y))

    test_ds = _make_ds(test_df).batch(batch_size)

    eval_out = model.evaluate(test_ds, return_dict=True, verbose=0)

    # metric names look like 'cls_auc', 'cls_pr_auc', 'score_rmse', …
    auc    = next(v for key, v in eval_out.items() if key.endswith("auc"))
    pr_auc = next(v for key, v in eval_out.items() if key.endswith("pr_auc"))
    bce    = eval_out.get("cls_loss", eval_out.get("loss", np.nan))

    # ---------- 2. sklearn-style classification metrics (0.5 cutoff) ---------- #
    # True labels
    y_true = test_df["label"].to_numpy().astype(int)

    # Predictions: need feats-only dataset for predict
    test_ds_feats_only = test_ds.map(lambda feats, y: feats)
    predictions = model.predict(test_ds_feats_only, verbose=0)

    # 'cls' probabilities
    y_prob = predictions["cls"].squeeze()  # shape (N,)
    y_pred = (y_prob > 0.5).astype(int)

    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)

    if print_reports:
        print("\n--- Classification Report (0.5 cutoff) ---")
        print(classification_report(y_true, y_pred, zero_division=0))
        print("--------------------------------------------------")
        if AltCutoff is not None and AltCutoff != 0.5:
            y_pred_alt = (y_prob >= AltCutoff).astype(int)
            print(f"Classification Report ({AltCutoff} cutoff) ---")
            print(classification_report(y_true, y_pred_alt, zero_division=0))
            print("--------------------------------------------------")

    # Start metrics dict with just the scalar stuff you care about
    metrics = {
        "auc":       auc,
        "pr_auc":    pr_auc,
        "bce":       bce,
        "accuracy":  acc,
        "precision": prec,
        "recall":    rec,
        "f1":        f1,
    }

    # # ---------- 3. Optional top-k retrieval metrics ---------- #
    # if compute_topk:
    #     true_sets, tot_pos = {}, 0
    #     for did, grp in test_df.groupby(disease_id_col):
    #         pos = set(grp.loc[grp.label == 1, "targetId"])
    #         if pos:
    #             true_sets[did] = pos
    #             tot_pos       += len(pos)

    #     hits = 0
    #     for _, row in disease_df.iterrows():
    #         did = row[disease_id_col]
    #         if did not in true_sets:
    #             continue
    #         preds = recommend_fn(
    #             did,
    #             row[disease_name_col],
    #             k=k,
    #             drop_known=drop_known,
    #         )
    #         hits += len(true_sets[did] & set(preds))

    #     prec_k = hits / (len(true_sets) * k)
    #     rec_k  = hits / tot_pos

    #     metrics[f"precision@{k}"] = prec_k
    #     metrics[f"recall@{k}"]    = rec_k

    if print_reports:
        print(
            f"\n★ Test AUC {auc:.4f} | PR-AUC {pr_auc:.4f} | BCE {bce:.3f}\n"
            # f"★ precision@{k} = {prec_k:.2%} | recall@{k} = {rec_k:.2%} "
            # f"over {len(true_sets)} diseases "
            f"({'dropping' if drop_known else 'including'} known positives)\n"
        )

    return metrics


In [ ]:
## code for manually getting all vs all preds (including pred score)

import tensorflow as tf
import pandas as pd
import numpy as np

def predict_all_global(model, disease_df, candidates_df, batch_size=2048, k=100):
    """
    Generates probabilities for ALL disease-candidate pairs using Matrix Multiplication.
    Returns raw top-k indices and scores to be formatted later.
    """
    print(f"1. Pre-computing embeddings for {len(candidates_df)} candidates...")
    
    # A. Encode Candidates (Using the model's exact training logic)
    #    We pass targetId even if the model ignores it, to match the signature.
    cand_embs = model.encode_k(
        candidates_df["target_text"].to_numpy(),
        candidates_df["targetId"].to_numpy() 
    )
    
    # B. Normalize Candidates (Matches Dot(normalize=True) from training)
    # cand_embs = tf.nn.l2_normalize(cand_embs, axis=1)
    cand_embs = tf.nn.l2_normalize(cand_embs, axis=1 , epsilon=1e-16) ## new added smaller epsilon-for more stability with small numbs

    print(f"2. Processing {len(disease_df)} diseases in batches...")
    results = []

    ### check col name of disease_text vs *_embed, rename if needed
    if "disease_text" not in disease_df.columns:
        if "disease_text_embed" in disease_df.columns:
            disease_df.rename(columns={"disease_text_embed":"disease_text"},inplace=True)
        else:
            print("error: disease_text (& disease_text_embed) missing")
            return None
    
    # Create dataset for batching queries (diseases)
    disease_ds = tf.data.Dataset.from_tensor_slices({
        "disease_text": disease_df["disease_text"].to_numpy(),
        "diseaseId": disease_df["diseaseId"].to_numpy()
    }).batch(batch_size)

    for batch in disease_ds:
        # C. Encode Queries
        q_embs = model.encode_q(batch["disease_text"], batch["diseaseId"])
        q_embs = tf.nn.l2_normalize(q_embs, axis=1) # Normalize
        
        # D. Matrix Multiplication (Cosine Similarity)
        #    Shape: (batch_size, num_candidates)
        sim_matrix = tf.matmul(q_embs, cand_embs, transpose_b=True)
        
        # E. Get Top-K raw similarities 
        #    (Much faster to sort raw scores than probas)
        top_k_sims, top_k_indices = tf.math.top_k(sim_matrix, k=k)
        
        # F. Convert Similarities to Probabilities 
        #    Pass raw cosine sim through the trained classification head
        flat_sims = tf.reshape(top_k_sims, (-1, 1))
        flat_probs = model.cls_head(flat_sims)
        top_k_probs = tf.reshape(flat_probs, tf.shape(top_k_sims))
        
        # Store results (DiseaseID, Candidate Indices, Probabilities)
        results.append((batch["diseaseId"].numpy(), top_k_indices.numpy(), top_k_probs.numpy()))
        print(f"   Processed batch...", end="\r")
        
    return results
    
def format_predictions(raw_results, candidates_df, positives_set, targets_map, top_n=5, min_prob=0.0):
    """
    Formats the raw tensors into a clean DataFrame, filtering known positives.
    Rounds probabilities to 5 decimal places.
    """
    print("\n3. Formatting and filtering...")
    data = []
    
    # Create a lookup for candidate IDs
    candidate_ids = candidates_df["targetId"].to_numpy()
    
    for batch_dids, batch_indices, batch_probs in raw_results:
        for i, disease_id_bytes in enumerate(batch_dids):
            disease_id = disease_id_bytes.decode('utf-8')
            
            found_ids = []
            found_syms = []
            found_probs = []
            
            # Iterate through the top-k candidates for this disease
            for rank_idx, cand_idx in enumerate(batch_indices[i]):
                target_id = candidate_ids[cand_idx]
                # --- FIX: Rounding here ---
                prob = round(float(batch_probs[i][rank_idx]), 3)
                
                # 1. Probability Filter
                if prob < min_prob:
                    continue
                    
                # 2. Novelty Filter (Remove known positives)
                # Note: pass positives_set=[] to see training data predictions
                # if len(positives_set)>0:
                if (disease_id, target_id) in positives_set:
                    continue
                
                # Add to list
                found_ids.append(target_id)
                found_probs.append(prob)
                found_syms.append(targets_map.get(target_id, {}).get("approvedSymbol", target_id))
                
                if len(found_ids) >= top_n:
                    break
            
            # Only add if we found valid predictions
            if found_ids:
                data.append({
                    "diseaseId": disease_id,
                    "novel_target_ids": found_ids,
                    "novel_target_sym": found_syms,
                    "probabilities": found_probs
                })
                
    return pd.DataFrame(data)

def explode_and_merge_positives(preds_df, positives_set, disease_df, targets_map):
    """
    1. Explodes the list-based predictions dataframe into long format.
    2. Creates a dataframe of known positives.
    3. Merges them into a single dataframe with a 'label' column.
    """
    # --- 0. Pre-process Input ---
    # Fix column naming mismatch (name -> diseaseName)
    if "name" in preds_df.columns:
        preds_df = preds_df.rename(columns={"name": "diseaseName"})
    
    # --- 1. Explode Predictions (Label = -1) ---
    print("Exploding predictions...")
    list_cols = ["novel_target_ids", "novel_target_sym", "probabilities"]
    
    # Check if input actually has lists (if empty, explode does nothing)
    if len(preds_df) > 0:
        long_preds = preds_df.explode(list_cols).rename(columns={
            "novel_target_ids": "targetId",
            "novel_target_sym": "targetSymbol",
            "probabilities": "score"
        })
    else:
        # Handle empty prediction case gracefully
        long_preds = pd.DataFrame(columns=["diseaseId", "diseaseName", "targetId", "targetSymbol", "score"])

    long_preds["label"] = -1
    long_preds["source"] = "model_prediction"
    
    cols_to_keep = ["diseaseId", "diseaseName", "targetId", "targetSymbol", "score", "label", "source"]
    # Ensure we only keep columns that exist (in case diseaseName was missing entirely)
    existing_cols = [c for c in cols_to_keep if c in long_preds.columns]
    long_preds = long_preds[existing_cols].copy()

    # --- 2. Create Known Positives DataFrame (Label = 1) ---
    print(f"Processing {len(positives_set)} known positives...")
    
    if len(positives_set) > 0:
        pos_df = pd.DataFrame(list(positives_set), columns=["diseaseId", "targetId"])
        
        # Map Disease Names
        dise_name_map = disease_df.set_index("diseaseId")["name"].to_dict()
        pos_df["diseaseName"] = pos_df["diseaseId"].map(dise_name_map)
        
        # Map Target Symbols
        def get_symbol(tid):
            return targets_map.get(tid, {}).get("approvedSymbol", tid)
        
        pos_df["targetSymbol"] = pos_df["targetId"].map(get_symbol)
        
        # Set Metadata
        pos_df["score"] = 1.0
        pos_df["label"] = 1
        pos_df["source"] = "known_positive"
        
        # Filter pos_df to cols_to_keep (handles any missing mappings)
        pos_df = pos_df.reindex(columns=cols_to_keep)
    else:
        pos_df = pd.DataFrame(columns=cols_to_keep)

    # --- 3. Merge and Clean ---
    print("Merging and cleaning...")
    combined_df = pd.concat([long_preds, pos_df], axis=0, ignore_index=True)
    
    # Drop duplicates: Prioritize Known Positive (label=1) over Prediction (label=-1)
    if not combined_df.empty:
        combined_df = combined_df.sort_values(
            by=["diseaseId", "targetId", "label"], 
            ascending=[True, True, False] # 1 comes before -1
        )
        combined_df = combined_df.drop_duplicates(subset=["diseaseId", "targetId"], keep="first")
    
    print(f"Final shape: {combined_df.shape}")
    return combined_df.sort_values(["diseaseId", "source","score"],ascending=False).reset_index(drop=True)

# --- usage ---
# all_candidates_long_df = explode_and_merge_positives(
#     final_preds_df, 
#     positives,       # The set of tuples {(did, tid), ...}
#     disease_df, 
#     targets_dict
# )
# display(all_candidates_long_df.head())

In [ ]:
def run_groupwise_cv(
    df_learn: pd.DataFrame,
    disease_df: pd.DataFrame,
    target_df: pd.DataFrame = None,
    n_splits: int = 5,
    n_repeats: int = 5,
    random_state: int = 42,
    epochs: int = 5,
    batch_size: int = 1024,
    k_recall: int = 10,
    drop_known: bool = False,
    val_frac: float = 0.01,
    AltCutoff: float = 0.65,
    EMB_ID: int = 64,
    return_oof_df: bool = False,   # NEW: optionally return out-of-fold preds
):
    """
    Repeated, groupwise CV by targetId.

    - Groups = targetId  (no leakage of targets between train / val / test)
    - Stratified on per-target label (any positive in that target -> label 1)
    - Uses n_splits folds, repeated n_repeats times (e.g. 5x5 CV)
    - Heavy data prep (text FeatureSpaces, disease lookup, candidate list)
      is done ONCE, then reused across repeats/folds.

    Returns
    -------
    results_df : pd.DataFrame
        One row per (repeat, fold) with metrics from `evaluate_model`.
    summary_df : pd.DataFrame
        Per-metric mean and std across all repeats × folds.
    oof_df : pd.DataFrame (only if return_oof_df=True)
        Same rows as df_learn, plus:
          - 'pred'      : mean out-of-fold predicted probability per row
          - 'oof_count' : how many times this row was in a test fold
    """
    if target_df is None:
        raise ValueError("run_groupwise_cv: target_df must be provided")

    # Work on a copy so we don't mutate the caller's df
    df_learn = df_learn.copy()

    # Stable row id for mapping predictions back, regardless of index changes
    if "_row_id" not in df_learn.columns:
        df_learn["_row_id"] = np.arange(len(df_learn), dtype=np.int64)

    # Storage for out-of-fold predictions (accumulate over all repeats × folds)
    if return_oof_df:
        oof_sum   = np.zeros(len(df_learn), dtype="float32")
        oof_count = np.zeros(len(df_learn), dtype="int32")

    # --- ensure text columns exist ---
    if "disease_text" not in df_learn.columns and "disease_text_embed" in df_learn.columns:
        df_learn.rename(columns={"disease_text_embed": "disease_text"}, inplace=True)
    if "target_text" not in df_learn.columns and "target_text_embed" in df_learn.columns:
        df_learn.rename(columns={"target_text_embed": "target_text"}, inplace=True)

    # --- 1. Per-target labels and groups (for stratified group CV) ---
    target_level = (
        df_learn[["targetId", "label"]]
        .groupby("targetId")["label"]
        .max()
        .reset_index()
    )
    tids = target_level["targetId"].to_numpy()
    y_target = target_level["label"].astype(int).to_numpy()

    # --- 2. One-time data prep: text FeatureSpaces & disease lookup ---
    # These do NOT depend on the particular fold and are reused.
    q_fs = make_fs()
    k_fs = make_fs()

    q_fs.adapt(
        tf.data.Dataset.from_tensor_slices({"text": df_learn["disease_text"]})
        .batch(2048)
        .prefetch(tf.data.AUTOTUNE)
    )
    k_fs.adapt(
        tf.data.Dataset.from_tensor_slices({"text": df_learn["target_text"]})
        .batch(2048)
        .prefetch(tf.data.AUTOTUNE)
    )

    dise_lookup = tf.keras.layers.StringLookup(name="disease_lookup")
    dise_lookup.adapt(df_learn["diseaseId"])

    # --- 3. Candidate pool: build once; embeddings will change per model ---
    druggable_genome_list = pd.read_csv(
        os.path.join("../data", "finan_proc_druggable_genome_list.csv")
    )["ensembl_gene_id"]

    candidates_df = (
        target_df[["targetId", "target_text_embed"]]
        .drop_duplicates(subset=["targetId"])
        .reset_index(drop=True)
        .rename(columns={"target_text_embed": "target_text"}, errors="ignore")
    )
    candidates_df = candidates_df.loc[
        (candidates_df["targetId"].isin(druggable_genome_list))
        | (candidates_df["targetId"].isin(df_learn["targetId"]))
    ].reset_index(drop=True)

    tid_lookup = tf.constant(candidates_df["targetId"].to_numpy())

    all_results = []

    # --- 4. Outer loop over repeats ---
    for rep in range(n_repeats):
        sgkf = StratifiedGroupKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=random_state + rep,
        )

        # --- 5. Inner loop over folds for this repeat ---
        for fold, (train_idx, test_idx) in enumerate(
            sgkf.split(X=np.zeros_like(y_target), y=y_target, groups=tids)
        ):
            train_tids = tids[train_idx]
            test_tids = tids[test_idx]

            # Map back to row-level df_learn
            train_rows = df_learn["targetId"].isin(train_tids)
            test_rows = df_learn["targetId"].isin(test_tids)

            df_train_full = df_learn.loc[train_rows].copy()
            df_test = df_learn.loc[test_rows].copy()

            # --- 5a. Inner validation split on training targets ---
            y_train_targets = y_target[train_idx]
            strat = y_train_targets if np.unique(y_train_targets).size > 1 else None

            inner_train_tids, val_tids = train_test_split(
                train_tids,
                test_size=val_frac,
                random_state=random_state + 1000 * rep + fold,
                shuffle=True,
                # stratify=strat, # disable ? 
            )

            train_mask = df_train_full["targetId"].isin(inner_train_tids)
            val_mask = df_train_full["targetId"].isin(val_tids)

            df_train = df_train_full.loc[train_mask].copy()
            df_val = df_train_full.loc[val_mask].copy()

            # Sanity: no target leaks between train / val / test
            assert set(inner_train_tids).isdisjoint(val_tids)
            assert set(inner_train_tids).isdisjoint(test_tids)
            assert set(val_tids).isdisjoint(test_tids)

            # --- 6. Build datasets for this fold ---
            train_ds = (
                make_ds(df_train)
                .shuffle(min(len(df_train), 150_000))
                .batch(batch_size)
                .prefetch(tf.data.AUTOTUNE)
            )
            val_ds = (
                make_ds(df_val)
                .batch(batch_size)
                .prefetch(tf.data.AUTOTUNE)
            )

            # --- 7. Fresh model for this fold (reuse q_fs/k_fs/dise_lookup) ---
            q_tower = build_tower(q_fs.get_encoded_features().shape[-1])
            k_tower = build_tower(k_fs.get_encoded_features().shape[-1] - EMB_ID)
            concat_layer = Concatenate(name=f"concat_cv_{rep}_{fold}")
            dise_emb = Embedding(
                dise_lookup.vocabulary_size(), EMB_ID, name=f"dise_emb_cv_{rep}_{fold}"
            )

            class TwoTowerDualCV(keras.Model):
                def __init__(self):
                    super().__init__()
                    self.dise_lookup = dise_lookup
                    self.dise_emb = dise_emb
                    self.q_fs, self.k_fs = q_fs, k_fs
                    self.q_tower, self.k_tower = q_tower, k_tower
                    self.concat = concat_layer
                    self.dot = keras.layers.Dot(axes=-1, normalize=True)
                    self.cls_head = keras.layers.Dense(
                        1, activation="sigmoid", name="cls",
                        bias_initializer=tf.keras.initializers.Constant(-2.1) # ADDED
                    )
                    self.score_head = keras.layers.Dense(
                        1,
                        activation=None,
                        name="score",
                        bias_initializer=tf.keras.initializers.Constant(
                            float(df_learn["score"].mean())
                        ),
                    )

                def encode_q(self, txt, did):
                    return self.q_tower(
                        self.concat(
                            [
                                self.q_fs({"text": txt}),
                                self.dise_emb(self.dise_lookup(did)),
                            ]
                        )
                    )

                def encode_k(self, txt, tid):
                    txt_vec = self.k_fs({"text": txt})
                    return self.k_tower(txt_vec)

                def call(self, feats):
                    q = self.encode_q(
                        feats["query"]["disease_text"],
                        feats["query"]["diseaseId"],
                    )
                    k = self.encode_k(
                        feats["candidate"]["target_text"],
                        feats["candidate"]["targetId"],
                    )
                    sim = self.dot([q, k])
                    prob = self.cls_head(sim)
                    reg = self.score_head(sim)
                    return {"cls": prob, "score": reg}

            model = TwoTowerDualCV()

            losses = {
                "cls": keras.losses.BinaryCrossentropy(from_logits=False),
                "score": keras.losses.MeanSquaredError(),
            }
            loss_weights = {"cls": 1.0, "score": 0.2}
            metrics_dict = {
                "cls": [
                    keras.metrics.AUC(name="auc"),
                    keras.metrics.AUC(curve="PR", name="pr_auc"),
                ],
                "score": [keras.metrics.RootMeanSquaredError(name="rmse")],
            }

            model.compile(
                optimizer=keras.optimizers.Adam(7e-3),
                loss=losses,
                loss_weights=loss_weights,
                metrics=metrics_dict,
            )

            callbacks = [
                keras.callbacks.ReduceLROnPlateau(
                    "val_cls_loss", mode="min", factor=0.2, patience=1
                ),
                keras.callbacks.EarlyStopping(
                    "val_cls_loss", mode="min", patience=2, restore_best_weights=False
                ),
            ]

            print(
                f"\n=== Repeat {rep+1}/{n_repeats}, fold {fold+1}/{n_splits} "
                f"(train targets={len(inner_train_tids)}, "
                f"val targets={len(val_tids)}, test targets={len(test_tids)}) ==="
            )
            model.fit(
                train_ds,
                validation_data=val_ds,
                epochs=epochs,
                callbacks=callbacks,
                verbose=1,
            )

            # --- 8. Build retrieval index for this fold ---
            cand_embs = model.encode_k(
                candidates_df["target_text"].to_numpy(),
                candidates_df["targetId"].to_numpy(),
            )
            retrieval = BruteForceRetrieval(k=k_recall, return_scores=True)
            retrieval.update_candidates(
                cand_embs, np.arange(len(candidates_df), dtype="int32")
            )

            if drop_known:
                train_pos = set(
                    zip(
                        df_train.query("label==1")["diseaseId"],
                        df_train.query("label==1")["targetId"],
                    )
                )
            else:
                train_pos = set()

            def recommend_fold(did, dtext, k, drop_flag):
                q = model.encode_q(
                    np.array([dtext]),
                    np.array([did]),
                )
                _, idx = retrieval(q)
                cand_ids = (
                    tf.gather(tid_lookup, idx[0]).numpy().astype(str).tolist()
                )
                if drop_flag:
                    cand_ids = [
                        t for t in cand_ids if (did, t) not in train_pos
                    ]
                return cand_ids[:k]

            # --- 9a. Accumulate out-of-fold predictions for this fold's test rows ---
            if return_oof_df:
                test_ds = (
                    make_ds(df_test)
                    .batch(batch_size)
                    .prefetch(tf.data.AUTOTUNE)
                )
                test_feats_ds = test_ds.map(lambda feats, y: feats)
                pred_dict = model.predict(test_feats_ds, verbose=0)
                y_prob = np.asarray(pred_dict["cls"]).reshape(-1)

                row_ids = df_test["_row_id"].to_numpy()
                oof_sum[row_ids] += y_prob
                oof_count[row_ids] += 1

            # --- 9b. Evaluate on this fold's test set (metrics only, no prints/top-k) ---
            metrics = evaluate_model(
                model=model,
                test_df=df_test,
                disease_df=disease_df,
                recommend_fn=lambda did, dname, k, drop_known: recommend_fold(
                    did, dname, k, drop_known
                ),
                k=k_recall,
                batch_size=batch_size,
                drop_known=drop_known,
                AltCutoff=AltCutoff,
                print_reports=False,
                compute_topk=False,
            )

            metrics["repeat"] = rep
            metrics["fold"] = fold
            all_results.append(metrics)

    results_df = pd.DataFrame(all_results)
    metric_cols = [c for c in results_df.columns if c not in ("repeat", "fold")]
    summary_df = (
        results_df[metric_cols]
        .agg(["mean", "std"])
        .T.reset_index()
        .rename(columns={"index": "metric"})
    )

    # --- 10. Optionally return out-of-fold prediction DataFrame ---
    if return_oof_df:
        # Avoid divide-by-zero; in practice every row should have oof_count>0
        denom = np.maximum(oof_count, 1)
        avg_pred = oof_sum / denom

        oof_df = df_learn.drop(columns=["_row_id"]).copy()
        oof_df["pred"] = avg_pred
        # oof_df["oof_count"] = oof_count

        return results_df, summary_df, oof_df

    # Backwards-compatible behavior
    return results_df, summary_df


# Code

In [ ]:
df_learn = pd.read_parquet("../data/proc/df_learn.parquet")
print(df_learn.shape)
display(df_learn)
disease_df = pd.read_parquet("../data/proc/disease_df.parquet")
print(disease_df.shape)
display(disease_df.head(2))
target_df = pd.read_parquet("../data/proc/target_df.parquet")
print(target_df.shape)
display(target_df.head(2))

In [ ]:
disease_df[["diseaseId","name"]].nunique()

Some targets are very common (housekeeping genes?)
* Could consider filtering them from predictions ?

In [ ]:
pos_target_counts = df_learn.query("label>0")["targetId"].value_counts()
display(pos_target_counts.describe(percentiles=[0.5,0.95,0.99]).round(1))
pos_target_counts.hist()

In [ ]:
df_learn.rename(columns={"disease_text_embed":"disease_text","target_text_embed":"target_text"},inplace=True,errors="ignore")
train_df = df_learn.copy()
display(train_df)

# ---------------------------------------------------------
# 1)  make a **target–hold-out** split  (70 % / 30 %)
# ---------------------------------------------------------
from sklearn.model_selection import train_test_split

# unique targets → split the *IDs* (not the rows)
train_tids, test_tids = train_test_split(
    df_learn["targetId"].unique(),
    test_size=0.2,          # paper: 70 % targets train, 30 % test
    random_state=42,
    shuffle=True,
    stratify = df_learn.drop_duplicates(subset=["targetId"])["label"]
)

train_df = df_learn[df_learn["targetId"].isin(train_tids)].copy()
test_df  = df_learn[df_learn["targetId"].isin(test_tids )].copy()

# ---------------------------------------------------------
# 2)  optional: split a *validation* set from the train targets
#     (again on targetId, 5 % of the training targets)
# ---------------------------------------------------------
train_tids, val_tids = train_test_split(
    train_tids,
    test_size=0.05,
    random_state=42,
    shuffle=True,
)

val_df   = train_df[train_df["targetId"].isin(val_tids)].copy()
train_df = train_df[train_df["targetId"].isin(train_tids)].copy()

In [ ]:
print("train/val/test target %")
print(train_df["label"].mean().round(2))
print(val_df["label"].mean().round(2))
print(test_df["label"].mean().round(2))

# --- quick sanity checks : that targetIDs are disjoint between train, test, val sets -------------------
assert set(train_tids).isdisjoint(val_tids),  "train & val targets overlap"
assert set(train_tids).isdisjoint(test_tids), "train & test targets overlap"
assert set(val_tids  ).isdisjoint(test_tids), "val & test targets overlap"

# optional: every original row is in exactly one split
assert len(train_df) + len(val_df) + len(test_df) == len(df_learn), \
       "row counts don't add up – some rows were lost or duplicated"

In [ ]:
## for use in init bias trick - faster convergence to class imbalance ? 
## https://www.tensorflow.org/tutorials/structured_data/imbalanced_data#examine_the_class_label_imbalance
neg, pos = np.bincount(df_learn['label'])
initial_bias = np.log([pos/neg])
initial_bias

In [ ]:
# ### make candidates

druggable_genome_list = pd.read_csv(os.path.join("../data", "finan_proc_druggable_genome_list.csv"))["ensembl_gene_id"]

## NEW: larger list, but of only druggable genome targets

candidates_df = (
    target_df[["targetId", "target_text_embed"]]      # add other target-side cols later
        .drop_duplicates(subset=["targetId"])  # pandas helper 
        .reset_index(drop=True).rename(columns={"target_text_embed":"target_text"},errors="ignore")
)
print(candidates_df.shape[0])
candidates_df = candidates_df.loc[(candidates_df["targetId"].isin(druggable_genome_list))|(candidates_df["targetId"].isin(df_learn["targetId"]))].reset_index(drop=True)
print(candidates_df.shape[0])

## DL

* Alt rewrite model (still 2 tower): https://www.tensorflow.org/recommenders/examples/featurization#putting_it_all_together
* * train/test split is disjoint on targets/targetId

### try 2 task model, predict also evidence score

* TODO : handle saving/serialization

In [ ]:
def make_ds(df: pd.DataFrame):
    feats = {
        "query": {
            "disease_text": df["disease_text"],
            "diseaseId"   : df["diseaseId"],
        },
        "candidate": {
            "target_text": df["target_text"],
            "targetId"   : df["targetId"],
        },
    }
    y = {
        "cls":   df["label" ].astype("float32"),   # -– 0 / 1
        "score": df["score"].astype("float32"),    # -– continuous
    }
    return tf.data.Dataset.from_tensor_slices((feats, y))
    

In [ ]:
# model = TwoTowerDual()
model = build_two_tower_model(df_learn) # df_learn vs train_df

In [ ]:
# model.save('dl_deep_model.keras',)
# loaded_model = keras.saving.load_model("dl_deep_model.keras")

```
def build_tower(input_dim: int,EMB_ID:int=64) -> keras.Model:
    inp = keras.Input(shape=(input_dim + EMB_ID,))
    out = keras.layers.Dense(512, activation="elu")(x)
    return keras.Model(inp, out, name="tower")


            self.cls_head    = keras.layers.Dense(1, activation="sigmoid", 
            name="cls",
            # 1. Start with a high scaling factor so Sigmoid isn't trapped in the middle.
        #    (This is trainable, so the model can lower it if 20 is too high).
        kernel_initializer=tf.keras.initializers.Constant(10.0),
            bias_initializer=tf.keras.initializers.Constant(-2.2))
```


```
def build_tower(input_dim: int, EMB_ID: int = 64) -> keras.Model:
    inp = keras.Input(shape=(input_dim + EMB_ID,))
    norm_x = keras.layers.LayerNormalization()(inp)

    # Path 1: The Linear Projection (Wide)
    linear_out = keras.layers.Dense(384, activation="linear")(norm_x)

    # Path 2: Non-linear capture (Optional complex interactions)
    deep = keras.layers.Dense(384, activation="elu")(norm_x)
    deep = keras.layers.LayerNormalization()(deep) # Norm inside deep block is fine
    deep = keras.layers.Dropout(0.35)(deep)
    
    deep = keras.layers.Dense(64, activation="elu")(deep)
    deep = keras.layers.Dropout(0.15)(deep)
    # # Remove the LN here if you are putting it at the end, 
    # # OR keep it if you want the deep branch specifically standardized.
    # # (Keeping it is fine/standard for a block).
    # deep = keras.layers.LayerNormalization()(deep)
    deep = keras.layers.Dense(384, activation="linear")(deep)

    # Add them (Residual style)
    out = keras.layers.Add()([linear_out, deep]) 
    # out = keras.layers.LayerNormalization(name="final_norm")(out)

    return keras.Model(inp, out, name="tower")

```

Classification Report (0.65 cutoff) ---
              precision    recall  f1-score   support

           0       0.96      0.99      0.97    116258
           1       0.92      0.65      0.76     14466

    accuracy                           0.95    130724
   macro avg       0.94      0.82      0.87    130724
weighted avg       0.95      0.95      0.95    130724

--------------------------------------------------

★ Test AUC 0.9454 | PR-AUC 0.8504 | BCE 0.148
(including known positives)


In [ ]:
losses = {
    "cls"  : keras.losses.BinaryCrossentropy(from_logits=False),
     # "cls2"  : keras.losses.BinaryFocalCrossentropy(apply_class_balancing=True),
    "score": keras.losses.MeanSquaredError(),   # or MAE / Huber
}

loss_weights = {"cls": 1.0, "score": 0.1} 

metrics = {
    "cls":  [keras.metrics.AUC(name="auc"),
             keras.metrics.AUC(curve="PR", name="pr_auc")],
    "score": [keras.metrics.RootMeanSquaredError(name="rmse")],
}

model.compile(optimizer=keras.optimizers.Adam(8e-3),
              loss=losses,
              loss_weights=loss_weights,
              metrics=metrics)

train_ds = make_ds(train_df).shuffle(3_00_000).batch(1024).prefetch(tf.data.AUTOTUNE)
val_ds   = make_ds(val_df).batch(2048).prefetch(tf.data.AUTOTUNE)

callbacks = [
    keras.callbacks.ReduceLROnPlateau("val_cls_loss", mode="min", factor=0.2, patience=1),
    keras.callbacks.EarlyStopping("val_cls_loss",   mode="min", patience=2, restore_best_weights=False),
]

if TRAIN_DL:    
    model.fit(train_ds, validation_data=val_ds, epochs=7, callbacks=callbacks)

with
```
    def build_tower(input_dim: int) -> keras.Model:
        inp = keras.Input(shape=(input_dim + EMB_ID,))
         x   = keras.layers.LayerNormalization()(inp)
         out = keras.layers.Dense(128, activation="gelu")(x)
        return keras.Model(inp, out, name="tower")
```
```
Epoch 10/10
247/247 ━━━━━━━━━━━━━━━━━━━━ 83s 336ms/step - cls_auc: 0.9687 - cls_loss: 0.1301 - cls_pr_auc: 0.8945 - loss: 0.1319 - score_loss: 0.0092 - score_rmse: 0.0957 - val_cls_auc: 0.9425 - val_cls_loss: 0.1273 - val_cls_pr_auc: 0.7547 - val_loss: 0.1311 - val_score_loss: 0.0066 - val_score_rmse: 0.0820 - learning_rate: 0.0016
```

test:

--- Classification Report (0.5 cutoff) ---
              precision    recall  f1-score   support

           0       0.96      0.98      0.97    116258
           1       0.78      0.69      0.73     14466

    accuracy                           0.94    130724


★ Test AUC 0.9502 | PR-AUC 0.8127 | BCE 0.142



```
def build_tower(input_dim: int, EMB_ID: int = 64) -> keras.Model:
    inp = keras.Input(shape=(input_dim + EMB_ID,))
    norm_x = keras.layers.LayerNormalization()(inp)

    # Path 1: The Linear Projection (What you have now - works well)
    linear_out = keras.layers.Dense(256, activation="linear")(norm_x)

    # Path 2: Non-linear capture (Optional complex interactions)
    deep = keras.layers.Dense(128, activation="elu")(norm_x)
    deep = keras.layers.Dropout(0.1)(deep)
    deep = keras.layers.Dense(128, activation="elu")(deep)
    # deep = keras.layers.Dropout(0.1)(deep)
    deep = keras.layers.Dense(256, activation="linear")(deep)

    # Add them (Residual style)
    out = keras.layers.Add()([linear_out, deep]) 
    
    return keras.Model(inp, out, name="tower")
```


        kernel_initializer=tf.keras.initializers.Constant(10.0),
            bias_initializer=tf.keras.initializers.Constant(-2.2))


<!-- ★ Test AUC 0.9482 | PR-AUC 0.8229 | BCE 0.161 -->


with 512; batch 4k
```
Epoch 11/11
124/124 ━━━━━━━━━━━━━━━━━━━━ 67s 540ms/step - cls_auc: 0.9982 - cls_loss: 0.0051 - cls_pr_auc: 0.9867 - loss: 0.0059 - score_loss: 0.0080 - score_rmse: 0.0897 - val_cls_auc: 0.9544 - val_cls_loss: 0.0110 - val_cls_pr_auc: 0.7985 - val_loss: 0.0123 - val_score_loss: 0.0066 - val_score_rmse: 0.0829 - learning_rate: 5.6000e-05
```

* with expanded text input fields:
```
Epoch 7/12
247/247 ━━━━━━━━━━━━━━━━━━━━ 70s 284ms/step - cls_auc: 0.9969 - cls_loss: 0.0151 - cls_pr_auc: 0.9762 - loss: 0.0159 - score_loss: 0.0082 - score_rmse: 0.0905 - val_cls_auc: 0.9528 - val_cls_loss: 0.0325 - val_cls_pr_auc: 0.7878 - val_loss: 0.0348 - val_score_loss: 0.0066 - val_score_rmse: 0.0833 - learning_rate: 0.0012
```
* with orig, smaller text fields features:
```
Epoch 6/12
494/494 ━━━━━━━━━━━━━━━━━━━━ 31s 62ms/step - cls_auc: 0.9970 - cls_loss: 0.0142 - cls_pr_auc: 0.9777 - loss: 0.0150 - score_loss: 0.0082 - score_rmse: 0.0904 - val_cls_auc: 0.9481 - val_cls_loss: 0.0352 - val_cls_pr_auc: 0.7625 - val_loss: 0.0378 - val_score_loss: 0.0067 - val_score_rmse: 0.0839 - learning_rate: 0.0012
```

Epoch 5/20
498/498 ━━━━━━━━━━━━━━━━━━━━ 26s 52ms/step - cls_auc: 0.9998 - cls_loss: 0.0021 - cls_pr_auc: 0.9986 - loss: 0.0029 - score_loss: 0.0078 - score_rmse: 0.0884 
- val_cls_auc: 0.9637 - val_cls_loss: 0.0158 - val_cls_pr_auc: 0.8948 - val_loss: 0.0168 - val_score_loss: 0.0088 - val_score_rmse: 0.0941 - learning_rate: 0.0018

##### TODO: Save + load model

In [ ]:
if TRAIN_DL:
    if SAVE_MODEL:
        # model.save('model.keras')  # <-- This is the problem
        model.save_weights('model.weights.h5') # <-- This is the fix
        print("Model weights saved to 'model.weights.h5'")## ERROR: twotowerdual 
        model.save('dl_retriever_model.keras')

In [ ]:
# model.save_weights('model_deep.weights.h5') # <-- This is the fix
# print("Model weights saved to 'model_deep.weights.h5'")## ERROR: twotowerdual 
# model.save('model_deep.keras')

In [ ]:
# # # Opt:Load the saved model
# if not TRAIN_DL:
#     try:
#         model = model.load_weights('model.weights.h5')
#         print("Successfully loaded model weights.")
#     except Exception as e:
#         print(f"Could not load model weights. Did you re-run the model definition cells? Error: {e}")

## extra eval on test set + get preds

##### TODO: Get "all druggable diseases" as targets for this + their text

In [ ]:
concat  = Concatenate(name="concat")
tid_lookup = tf.constant(candidates_df["targetId"].to_numpy())  # shape (N,)

# ------------- create a set of known (diseaseId, targetId) positives
positives = set(zip(df_learn.query("label==1")["diseaseId"],
                    df_learn.query("label==1")["targetId"]))

targets_dict = target_df.reset_index().set_index("targetId")[["approvedSymbol","approvedName"]].to_dict(orient="index") # map IDs to readable names for 17k~ genes/targets
    

In [ ]:
### Following doesn't work after changing model saving/serialization structure
"""
# ════════════════════════════════════════════════════════════
# 6.  Retrieval index
# ════════════════════════════════════════════════════════════
# cand_vec  = concat([k_fs({"text": candidates_df["target_text"]}),
#                     targ_emb(targ_lookup(candidates_df["targetId"]))])
if TRAIN_DL:
    cand_vec  = concat([k_fs({"text": candidates_df["target_text"]})])
    cand_embs = k_tower(cand_vec)
    
    retrieval = BruteForceRetrieval(k=30, return_scores=True)
    retrieval.update_candidates(cand_embs,
                                np.arange(len(candidates_df), dtype="int32"))
    
    # # Example predictions
    # qry_embed = model.encode_q(np.array(["lung fibrosis"]),
    #                            np.array(["MONDO_0000160"]))
    # _, idx = retrieval(qry_embed)
    # print("Top‑10 targets:",
    #       tf.gather(candidates_df["targetId"].to_numpy(), idx[0]).numpy().tolist())
    ## reccommend and filter out known positives
    # -------------------------------------------------------------------
    # 1.  Build an int→string lookup tensor  (once)
    # -------------------------------------------------------------------
    # tid_lookup = tf.constant(candidates_df["targetId"].to_numpy())  # shape (N,)
    
    # # ------------- create a set of known (diseaseId, targetId) positives
    # positives = set(zip(df_learn.query("label==1")["diseaseId"],
    #                     df_learn.query("label==1")["targetId"]))
    ######
    def recommend(disease_id, disease_text, k=7, drop_known=True):
        q = model.encode_q(np.array([disease_text]), np.array([disease_id]))
        _, idx = retrieval(q)
        cand = tf.gather(tid_lookup, idx[0]).numpy().astype(str)
        if drop_known:
            cand = [t for t in cand if (disease_id, t) not in positives]
        return cand[:k]
    
    # -------------------------------------------------------------------
    # 2.  Pretty-print an example
    # -------------------------------------------------------------------
    # targets_dict = target_df.reset_index().set_index("targetId")[["approvedSymbol","approvedName"]].to_dict(orient="index") # map IDs to readable names for 17k~ genes/targets
    
    example_id   = "MONDO_0000160"
    example_text = "lung fibrosis" # needs actual disease text, nvm more features later
    
    print("Top-10 *novel* targets for", example_text, ":")
    # print("\n".join(recommend(example_id, example_text, drop_known=True)))
    res = recommend(example_id, example_text, drop_known=True)
    print([*map(targets_dict.get, res)])
    print("\nMONDO_0100233 - long Covid:")
    # res = recommend("MONDO_0100233", " long haul COVID-19. post-acute sequelae of COVID-19. A chronic disease triggered by acute COVID-19 infection")
    example_id, example_text = disease_df.loc[disease_df["diseaseId"]=="MONDO_0100233"][["diseaseId","disease_text_embed"]].iloc[0].values
    res = recommend(example_id, example_text, drop_known=True)
    # print("\n".join(recommend("MONDO_0100233", " long haul COVID-19. post-acute sequelae of COVID-19. A chronic disease triggered by acute COVID-19 infection")))
    print([*map(targets_dict.get, res)])

    print("\n diabetes mellitus type 2 associated cataract")
    example_id, example_text = disease_df.loc[disease_df["name"]=="diabetes mellitus type 2 associated cataract"][["diseaseId","disease_text_embed"]].iloc[0].values
    res = recommend(example_id, example_text, drop_known=True)
    print([*map(targets_dict.get, res)])


    ## Essential hypertension
    print("\nEssential hypertension")
    example_id, example_text = disease_df.loc[disease_df["name"]=="essential hypertension"][["diseaseId","disease_text_embed"]].iloc[0].values
    res = recommend(example_id, example_text, drop_known=True)
    print([*map(targets_dict.get, res)])

    print("\nMultiple sclerosis")
    example_id, example_text = disease_df.loc[disease_df["name"]=="multiple sclerosis"][["diseaseId","disease_text_embed"]].iloc[0].values
    res = recommend(example_id, example_text, drop_known=True)
    print([*map(targets_dict.get, res)])
    
    print("\nAsperger syndrome/Autism")
    example_id, example_text = disease_df.loc[disease_df["name"]=="Asperger syndrome"][["diseaseId","disease_text_embed"]].iloc[0].values
    res = recommend(example_id, example_text, drop_known=True)
    print([*map(targets_dict.get, res)])    

    print("\ntreatment refractory schizophrenia") # Schizophrenia which does not respond to common
    example_id, example_text = disease_df.loc[disease_df["name"]=="treatment refractory schizophrenia"][["diseaseId","disease_text_embed"]].iloc[0].values
    res = recommend(example_id, example_text, drop_known=True)
    print([*map(targets_dict.get, res)])    
"""

def recommend(disease_id, disease_text, k=7, drop_known=True):
    q = model.encode_q(np.array([disease_text]), np.array([disease_id]))
    _, idx = retrieval(q)
    cand = tf.gather(tid_lookup, idx[0]).numpy().astype(str)
    if drop_known:
        cand = [t for t in cand if (disease_id, t) not in positives]
    return cand[:k]

## Evaluate - test-set

In [ ]:
if TRAIN_DL:
    try:
        metrics = evaluate_model(
            model        = model,
            test_df      = test_df,
            disease_df   = disease_df.reset_index()[["diseaseId", "name"]],
            # recommend_fn = lambda did, dname, k, drop_known:
            #                   recommend(did, dname, k=k, drop_known=drop_known),
            # # k            = 5,
            drop_known   = False,      # or True
        AltCutoff=0.65,
        )
    except Exception as e:
        # Handle the exception
        print(f"An error occurred: {e}")

wide and deep:
```

--- Classification Report (0.5 cutoff) ---
              precision    recall  f1-score   support

           0       0.96      0.99      0.98    116258
           1       0.87      0.71      0.78     14466

    accuracy                           0.96    130724
   macro avg       0.92      0.85      0.88    130724
weighted avg       0.95      0.96      0.95    130724

--------------------------------------------------
Classification Report (0.65 cutoff) ---
              precision    recall  f1-score   support

           0       0.96      0.99      0.98    116258
           1       0.91      0.66      0.77     14466

    accuracy                           0.96    130724
   macro avg       0.93      0.83      0.87    130724
weighted avg       0.95      0.96      0.95    130724

--------------------------------------------------

★ Test AUC 0.9468 | PR-AUC 0.8525 | BCE 0.149
```

```relu model
★ Test AUC 0.9451 | PR-AUC 0.8339 | BCE 0.156

1       0.90      0.63      0.74     14466
```

```
--------------------------------------------------
Classification Report (0.6 cutoff) ---
              precision    recall  f1-score   support

           0       0.96      0.99      0.97    116258
           1       0.89      0.63      0.74     14466

    accuracy                           0.95    130724
   macro avg       0.92      0.81      0.86    130724
weighted avg       0.95      0.95      0.95    130724

--------------------------------------------------

★ Test AUC 0.9438 | PR-AUC 0.8326 | BCE 0.143
```

★ Test AUC 0.9433 | PR-AUC 0.8043 | BCE 0.0461

## Evaluate baselines

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    accuracy_score,
    precision_score,
    recall_score,
)

def evaluate_prob_series(y_true, y_hat, name, threshold=0.5):
    # Handle epsilon vs eps for log_loss
    ll_kwargs = {"labels": [0, 1]}
    param_names = log_loss.__code__.co_varnames
    if "eps" in param_names:
        ll_kwargs["eps"] = 1e-7
    elif "epsilon" in param_names:
        ll_kwargs["epsilon"] = 1e-7

    # Binary predictions from scores
    y_pred_bin = (y_hat >= threshold).astype("int32")

    print(
        f"{name:18s}  "
        f"AUC={roc_auc_score(y_true, y_hat):.4f}  "
        f"PR-AUC={average_precision_score(y_true, y_hat):.4f}  "
        f"BCE={log_loss(y_true, y_hat, **ll_kwargs):.4f}  "
        f"Acc={accuracy_score(y_true, y_pred_bin):.4f}  "
        f"P={precision_score(y_true, y_pred_bin, zero_division=0):.4f}  "
        f"R={recall_score(y_true, y_pred_bin):.4f}"
    )

def target_mean_baseline(test_df, full_df):
    """Return target-wise oracle mean predictions from full dataset (df_learn)."""
    target_means = full_df.groupby("targetId")["label"].mean()
    return test_df["targetId"].map(target_means).fillna(target_means.mean()).to_numpy(dtype="float32")

y_test = test_df["label"].to_numpy(dtype="float32")

evaluate_prob_series(y_test,
                     test_df[["diseaseId"]].merge(train_df.groupby("diseaseId")["label"].mean(),on="diseaseId",how="left")["label"].fillna(0), # mean of disease in train, apply to test
                     "DISEASE mean")

### code for these baselines was deleted
# evaluate_prob_series(y_test,
#                      constant_baseline(test_df, train_df["label"].mean()),
#                      "GLOBAL")

# evaluate_prob_series(y_test,
#                      disease_mean_baseline(test_df, train_df),
#                      "DISEASE mean")


In [ ]:
evaluate_prob_series(y_test,
                     test_df[["diseaseId"]].merge(train_df.groupby("diseaseId")["label"].mean(),on="diseaseId",how="left")["label"].fillna(train_df["label"].mean()), # mean of disease in train, apply to test
                     "DISEASE mean")

### dig into reccs

* add implicit learning task / multitask learning
*  Add Factorized-Top-K ranking metrics ( + optional multi-task training )

In [ ]:
# if TRAIN_DL:
#     print("post-training")
#     ## train additional epoch on val+test data
#     # model.fit(make_ds(df_learn).shuffle(1_00_000).batch(2048).prefetch(tf.data.AUTOTUNE), epochs=1)
    
#     model.fit(make_ds(pd.concat([test_df.query("label>0"),val_df.query("label>0"),train_df],ignore_index=True).sample(frac=1)
#                      ).shuffle(1_00_000).batch(2048).prefetch(tf.data.AUTOTUNE), epochs=1)

In [ ]:
# 1. Identify "Orphan Diseases" (<= 1 known positive target in df_learn)
print("1. Identifying orphan diseases (<= 1 known positive target)...")
# Summing 'label' column (0 or 1) gives the total positive targets
disease_pos_counts = df_learn.groupby("diseaseId")["label"].sum()

# Get the set of disease IDs with 0  positive targets
orphan_disease_ids = set(disease_pos_counts[disease_pos_counts <= 0].index)
print(f"Found {len(orphan_disease_ids)} orphan diseases (out of {len(disease_pos_counts)} total).")

# 2. Identify "Hub Proteins" (>= 200 associated diseases in df_learn)
print("\n2. Identifying hub proteins (>= 200 associated diseases)...")
# Use the same logic as cell [26] to count positive links for targets 
pos_target_counts = df_learn.query("label > 0")["targetId"].value_counts()

# Get the set of target IDs with 200 or more links
HUB_THRESHOLD = 200
hub_protein_ids = set(pos_target_counts[pos_target_counts >= HUB_THRESHOLD].index)
print(f"Found {len(hub_protein_ids)} hub proteins at threshold >={HUB_THRESHOLD}.")
if len(hub_protein_ids) > 0:
    print(f"  Example hubs found: {list(hub_protein_ids)[:5]}")

## OPT: Get all (novel) predictions


In [ ]:
%%time
if GET_ALL_PREDS:
    # 1. Define the diseases you want to predict for
    #    (Using your 'orphan' logic: diseases in df_learn with <= 2 known targets)
    #    Ensure we use the disease_df that has the embeddings
    target_subset = disease_df.loc[
       # (~disease_df["diseaseId"].isin(orphan_disease_ids)) & 
        disease_df["diseaseId"].isin(df_learn["diseaseId"])
    ]
    
    print(f"Generating predictions for {len(target_subset)} diseases...")

    # 2. Run the robust prediction
    #    We ask for top-50 initially to have a buffer for filtering known positives
    raw_results = predict_all_global(model, target_subset, candidates_df, k=800)
    
    # 3. Format, Filter Known Positives, and Threshold
    final_preds_df = format_predictions(
        raw_results, 
        candidates_df, 
        positives
        ,targets_dict, 
        top_n=200, 
        min_prob=0.65 # <-- Set  probability threshold
    )
    if len(final_preds_df)>0:
        # 4. Merge names for readability
        final_preds_df = final_preds_df.merge(disease_df[["diseaseId", "name"]], on="diseaseId", how="left")
        
        print(final_preds_df.shape[0],"# diseases with candidates")
        display(final_preds_df)

        #     # # Save
        # final_preds_df.to_csv("DL_novel_predictions_wide.csv", index=False)
    else:
        print("\nNo cases!!")

In [ ]:
if GET_ALL_PREDS:
    if len(final_preds_df)>0:
        # --- usage ---
        all_candidates_long_df = explode_and_merge_positives(
            final_preds_df, 
            positives,       # The set of tuples {(did, tid), ...}
            disease_df, 
            targets_dict
        )
        
        # add known clinical targets (if any) per disease
        all_candidates_long_df["disease_num_known_clinical_targets"] = all_candidates_long_df["diseaseId"].map(disease_pos_counts)
        all_candidates_long_df["orphan"] = all_candidates_long_df["diseaseId"].isin(orphan_disease_ids)
        
        print("Novel nunique:\n",all_candidates_long_df.query("label== -1")[['diseaseId', 'diseaseName', 'targetId']].nunique())
        display(all_candidates_long_df["source"].value_counts())
        print("# mean score by ground truth/preds:")
        display(all_candidates_long_df.groupby(["source"])["score"].mean())
        print("#nunique")
        display(all_candidates_long_df[["diseaseId","targetId"]].nunique())
        print("Orphan diseases preds:",all_candidates_long_df.drop_duplicates("diseaseId").groupby(["source"])["orphan"].sum())
        display(all_candidates_long_df)
        
        if SAVE_PREDS:
            # Save
            all_candidates_long_df.to_csv("./Outputs/DL_novel_candidates_predictions.csv", index=False)
            all_candidates_long_df.query("label== -1")[['diseaseId', 'diseaseName', 'targetId', 'targetSymbol', 'score',
                'disease_num_known_clinical_targets', 'orphan']].to_csv("./Outputs/DL_novel_predictions.csv", index=False)

In [ ]:
# ## look at cases with known targets , and don't filter out positives:
# target_subset = disease_df[
#    (~disease_df["diseaseId"].isin(orphan_disease_ids)) & 
#     disease_df["diseaseId"].isin(df_learn["diseaseId"])
# ]

# print(f"Generating predictions for {len(target_subset)} diseases...")

# # 2. Run the robust prediction
# raw_results = predict_all_global(model, target_subset, candidates_df, k=150)

# # 3. Format, Filter Known Positives, and Threshold
# final_preds_df = format_predictions(
#     raw_results, 
#     candidates_df, 
#     # positives
#     []  # positives
#     ,targets_dict, 
#     top_n=3, 
#     min_prob=0.06 # <-- probability threshold
# )

# # 4. Merge names for readability
# final_preds_df = final_preds_df.merge(disease_df[["diseaseId", "name"]], on="diseaseId", how="left")
# print(final_preds_df.shape[0],"# diseases with candidates")
# display(final_preds_df)


In [ ]:
all_candidates_long_df.query("label== -1")["targetSymbol"].value_counts()

In [ ]:
all_candidates_long_df.query("label== -1")["diseaseName"].value_counts()

##### OPT: Filter out for predictions on orphan diseases + exclude hub/common genes

## CV eval
* SD
* splits by target
  opt? get pred score per data point?

In [ ]:
if RUN_CV: ## clear memory
    import gc
    # del model
    tf.keras.backend.clear_session()
    gc.collect()

In [ ]:
%%time
if RUN_CV:
    results_df, summary_df, oof_df = run_groupwise_cv(
        df_learn=df_learn
        ,disease_df=disease_df,
        target_df=target_df,
        n_splits=5,
        n_repeats=5,
        epochs=10,
        batch_size=512,
        drop_known=False,
        val_frac=0.01,
        AltCutoff=0.65,
        EMB_ID=EMB_ID,
        return_oof_df=True,  # <– this is the only new argument you need
    )
    
    # For ROC/PR curves:
    y_true = oof_df["label"].to_numpy(dtype="float32")
    y_pred = oof_df["pred"].to_numpy(dtype="float32")
    
    # print(roc_auc_score(y_true, y_pred))
    # print(average_precision_score(y_true, y_pred))

In [ ]:
if RUN_CV:
    print(classification_report(y_true, y_pred>0.5))
    print("--"*10)
    print("ROCAUC:",roc_auc_score(y_true, y_pred))
    print("PRAUC:",average_precision_score(y_true, y_pred))

5x5 cv dl results:

    * roc_auc_score: 0.95979936
    * average_precision_score (same as prauc): 0.84289

Wide and deep model (CV):
```
              precision    recall  f1-score   support

         0.0       0.97      0.99      0.98    595819
         1.0       0.88      0.70      0.78     67532

    accuracy                           0.96    663351
   macro avg       0.92      0.84      0.88    663351
weighted avg       0.96      0.96      0.96    663351

--------------------
ROCAUC: 0.962505
PRAUC: 0.859746
```

In [ ]:
if RUN_CV:
    print(classification_report(y_true, y_pred>=0.65))

In [ ]:
if RUN_CV:
    display(summary_df)
    display(results_df)

In [ ]:
results_df.agg(["mean","std"]).T

In [ ]:
if RUN_CV:
    # if SAVE_PREDS:
    # oof_df.drop(columns=["oof_count"]).to_csv("oof_dl_preds.csv",index=False) # parquet will be smaller
    oof_df.drop(columns=["oof_count"],errors="ignore").to_parquet("./Outputs/CV_dl/oof_dl_preds.parquet",index=False)

    summary_df.to_csv("./Outputs/CV_dl/oof_dl_summary.csv",index=False)
    results_df.to_csv("./Outputs/CV_dl/oof_dl_results.csv",index=False)

#### Check OOF Preds correltation with clinical stage reached (positives only)
We would expect high confidence predictions to make it to a later clinical trial stage. 
* Spearman rho = 0.1997
* Pearson r   = 0.18

In [ ]:
# Merge target prioritisation (max clinical trial phase) with OOF preds,
import re, numpy as np, math
from scipy.stats import spearmanr, pearsonr, norm
import matplotlib.pyplot as plt
import seaborn as sns

# Load target prioritisation (use copy_proc fallback)
tp_path = './copy_proc/target_prioritisation.parquet'
if not os.path.exists(tp_path):
    raise FileNotFoundError(f'Expected target prioritisation at {tp_path}')
tp = pd.read_parquet(tp_path)
print('Loaded target prioritisation with columns:', tp.columns.tolist())
# Ensure targetId column present
if 'id' in tp.columns and 'targetId' not in tp.columns:
    tp = tp.rename(columns={'id':'targetId'})
# Identify clinical phase column (common name: maxClinicalTrialPhase)
phase_col = None
cands = [c for c in tp.columns if 'clinical' in c.lower() or 'phase' in c.lower()]
if 'maxClinicalTrialPhase' in tp.columns:
    phase_col = 'maxClinicalTrialPhase'
elif cands:
    phase_col = cands[0]
else:
    raise KeyError('No clinical phase-like column found in target prioritisation')
print('Using phase column:', phase_col)
tp_phase = tp[[ 'targetId', phase_col ]].copy()

# Ensure oof_df available (try loading saved preds otherwise)
try:
    oof = oof_df.copy()
except NameError:
    fallback = './Outputs/CV_DL/oof_dl_preds.parquet'
    if os.path.exists(fallback):
        oof = pd.read_parquet(fallback)
    else:
        raise NameError('oof_df not in memory and fallback parquet not found')

print('OOF dataframe shape:', oof.shape)
# Prediction column: prefer 'pred' then 'score' then any numeric column named like 'prob'
if 'pred' in oof.columns:
    pred_col = 'pred'
elif 'score' in oof.columns:
    pred_col = 'score'
else:
    # pick first float column that's not label/target/disease
    numcols = oof.select_dtypes(include=[np.number]).columns.tolist()
    forbidden = {'label'}
    cand_cols = [c for c in numcols if c not in forbidden]
    if cand_cols:
        pred_col = cand_cols[0]
    else:
        raise KeyError('No prediction numeric column found in oof_df')
print('Using prediction column:', pred_col)

# Merge and filter positives only
dfm = oof.merge(tp_phase, on='targetId', how='left')
df_pos = dfm[dfm.get('label', pd.Series()).fillna(0) == 1].copy()
print('Positives in OOF after merge:', df_pos.shape)

# Normalize/convert phase values to numeric (e.g. 'Phase 1' -> 1, 'preclinical'->0)
def phase_to_num(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float)):
        return x
    s = str(x)
    s_low = s.lower()
    if 'pre' in s_low:
        return 0
    m = re.search(r'phase\s*[:\-\s]?(\d+)', s, re.I)
    if m:
        return int(m.group(1))
    # also catch lone integers
    m2 = re.search(r'^(\d+)$', s.strip())
    if m2:
        return int(m2.group(1))
    return np.nan

df_pos['phase_num'] = df_pos[phase_col].apply(phase_to_num)
df_pos = df_pos.dropna(subset=['phase_num'])
df_pos['phase_num'] = df_pos['phase_num'].astype(float)
print('After dropping NA phases, rows:', len(df_pos))

def format_p_with_fallback(p, coef, n, test='pearson'):
    # If p is nonzero and finite, format normally
    if p and np.isfinite(p) and p > 0:
        return f'{p:.3e}'
    # Fallback: approximate tail using normal approximation to avoid underflow and show a bound
    try:
        if test == 'pearson':
            # approximate z from r (Pearson) using t->z style transform
            z = coef * math.sqrt((n - 2) / (1.0 - coef * coef))
        else:
            # spearman approx: rho * sqrt(n-1)
            z = coef * math.sqrt(max(n - 1, 1))
        # two-sided p approx from normal tail in log space
        log_tail = norm.logsf(abs(z))
        log_p = log_tail + math.log(2.0)
        # convert to exponent base-10
        log10p = log_p / math.log(10)
        exp = int(max(1, math.floor(-log10p)))
        return f'<1e-{exp}'
    except Exception:
        return '0'

if len(df_pos) == 0:
    print('No positives with non-null clinical phase found — nothing to correlate')
else:
    # Compute correlations
    x = df_pos['phase_num'].to_numpy()
    y = df_pos[pred_col].to_numpy()
    sp_coef, sp_p = spearmanr(x, y)
    pe_coef, pe_p = pearsonr(x, y)
    sp_p_fmt = format_p_with_fallback(sp_p, sp_coef, len(x), test='spearman')
    pe_p_fmt = format_p_with_fallback(pe_p, pe_coef, len(x), test='pearson')
    print(f'Spearman rho = {sp_coef:.4f}, p = {sp_p_fmt}')
    print(f'Pearson r   = {pe_coef:.4f}, p = {pe_p_fmt}')

    # Summary: mean pred per phase
    summary = df_pos.groupby('phase_num')[pred_col].agg(['count','mean','median']).reset_index().sort_values('phase_num')
    display(summary)

    # Boxplot by phase and scatter
    plt.figure(figsize=(8,4))
    sns.boxplot(x='phase_num', y=pred_col, data=df_pos)
    plt.xlabel('Clinical phase (numeric)')
    plt.ylabel('OOF prediction')
    plt.title(f'OOF prediction by clinical phase (n={len(df_pos)})')
    plt.show()

    plt.figure(figsize=(6,4))
    sns.regplot(x='phase_num', y=pred_col, data=df_pos, scatter_kws={'alpha':0.5}, ci=None)
    plt.xlabel('Clinical phase (numeric)')
    plt.ylabel('OOF prediction')
    plt.title(f'Scatter + linear fit — Pearson r={pe_coef:.3f}, p={pe_p_fmt}')
    plt.show()